# Semana 6 · Sesión 1: Álgebra lineal simbólica

**Módulo 1**

## Objetivos de la sesión

1. Construir y operar matrices simbólicas con `sp.Matrix`, sabiendo qué
   devuelven `det`, `inv`, `rank` y la transpuesta.
2. Escribir un sistema lineal en forma matricial $A\mathbf{x} = \mathbf{b}$ y
   resolverlo, entendiendo qué significa físicamente que $\det A = 0$.
3. Calcular eigenvalores y eigenvectores simbólicos, y leerlos como las
   frecuencias y los modos normales de un sistema de osciladores acoplados.

## Retomamos

La semana pasada resolvimos sistemas de ecuaciones con `linsolve` y ecuaciones
diferenciales con `dsolve`. La máquina de Atwood salió de dos ecuaciones
lineales; el oscilador armónico, de una ecuación diferencial con condiciones
iniciales.

Hoy juntamos las dos cosas. Cuando hay **varios** grados de libertad acoplados
—dos masas unidas por resortes, un cuerpo rígido girando— las ecuaciones dejan
de poder escribirse una por una de forma cómoda, y la herramienta natural es la
matriz. Y lo mejor: los eigenvalores de esa matriz no son un tecnicismo
algebraico, son **frecuencias**.

Este notebook es autocontenido: la celda de abajo declara todo lo que necesita.

In [ ]:
import sympy as sp

sp.init_printing()

# Símbolos del día, con las suposiciones que la física garantiza.
t = sp.Symbol("t", real=True)
m, m1, m2 = sp.symbols("m m1 m2", positive=True)   # masas
k, g = sp.symbols("k g", positive=True)            # constante elástica y gravedad
a, b, c, d = sp.symbols("a b c d")                 # entradas genéricas

## `Matrix`: construir una matriz

Una matriz de SymPy se escribe como una **lista de renglones**, cada renglón a
su vez una lista. Hay atajos para las matrices que uno usa todo el tiempo:

| Quieres | Escribe |
|---|---|
| Matriz explícita | `sp.Matrix([[1, 2], [3, 4]])` |
| Identidad $n\times n$ | `sp.eye(n)` |
| Matriz de ceros | `sp.zeros(filas, columnas)` |
| Diagonal con entradas dadas | `sp.diag(m1, m2)` |
| Vector columna | `sp.Matrix([x, y])` |

Un vector columna es simplemente una matriz de una sola columna: en SymPy no
hay un tipo aparte para vectores.

In [ ]:
A = sp.Matrix([[a, b], [c, d]])

display(A)
display(sp.eye(2))
display(sp.diag(m1, m2))     # la matriz de masas del final de la sesión
display(sp.Matrix([m*g, 0])) # un vector columna

## Leer y modificar: `shape`, indexado y rebanadas

El tamaño está en `.shape`, una tupla de Python. El indexado es `M[fila,
columna]`, contando desde cero, y admite rebanadas: `M[:, 0]` es la primera
columna, `M[1, :]` el segundo renglón.

Y hay una diferencia importante con los objetos de SymPy que veníamos usando:
`sp.Matrix` **sí es mutable** — `M[0, 0] = ...` funciona. Es la excepción, y
existe por comodidad al armar matrices grandes. `subs`, en cambio, sigue siendo
inmutable: devuelve una matriz nueva y deja la original intacta.

In [ ]:
print("shape:", A.shape)     # una tupla de Python, no una expresión

display(A[0, 1])             # la entrada b
display(A[:, 0])             # la primera columna

# subs devuelve una matriz nueva; A no cambia.
display(A.subs({a: 1, d: 1}))
display(A)

## Operaciones: el asterisco es producto matricial

Suma, resta y producto se escriben con los operadores de siempre, con una
trampa que hay que decir en voz alta: **`*` es el producto matricial**, no el
producto entrada por entrada. Quien venga de NumPy tiene que desaprender esto.

| Operación | Escribe |
|---|---|
| Producto matricial | `A * B` |
| Potencia (matricial) | `A**2` |
| Transpuesta | `A.T` |
| Determinante | `A.det()` |
| Inversa | `A.inv()` |
| Rango | `A.rank()` |
| Producto entrada por entrada | `sp.matrix_multiply_elementwise(A, B)` |

Un detalle de presentación que vamos a usar toda la sesión: para etiquetar una
matriz con su nombre —`sp.Eq(sp.Symbol("K"), rigidez)`— hay que pasar
`evaluate=False`. Sin eso, `sp.Eq` compara el símbolo con la matriz, ve que no
son el mismo objeto y devuelve `False`. Es la misma trampa de la semana 5 con
la integral y su valor numérico.

In [ ]:
B = sp.Matrix([[0, 1], [1, 0]])   # intercambia los renglones

display(A * B)
display(B * A)

# El producto de matrices no conmuta, y aquí se ve: la diferencia no es cero.
display(sp.Eq(sp.Symbol("AB - BA"), sp.simplify(A*B - B*A), evaluate=False))

## Determinante, inversa y rango

`det` e `inv` son simbólicos: devuelven fórmulas, no números. La inversa lleva
el determinante en el denominador, así que **una matriz simbólica siempre se
deja invertir**, y el problema aparece después, al sustituir valores que anulan
ese determinante.

Por eso el determinante merece una mirada antes de invertir nada: en un sistema
físico, $\det A = 0$ es la frontera entre "hay una solución única" y "el sistema
está degenerado" — una configuración de equilibrio indeterminada, un mecanismo
que se puede mover sin costo, una frecuencia de resonancia.

In [ ]:
display(sp.Eq(sp.Symbol("det(A)"), A.det()))
display(A.inv())

print("rank:", A.rank())                      # un entero de Python
print("rank de una singular:", sp.Matrix([[1, 2], [2, 4]]).rank())

## Sistemas lineales en forma matricial

Un sistema lineal siempre puede escribirse como $A\mathbf{x} = \mathbf{b}$: la
matriz $A$ guarda los coeficientes, el vector $\mathbf{b}$ los términos
independientes. Resolverlo es `A.solve(b)`.

Retomemos la **máquina de Atwood** de la semana pasada. Las dos ecuaciones eran

$$m_1 a = m_1 g - T, \qquad m_2 a = T - m_2 g$$

Ordenadas con las incógnitas $a$ y $T$ del lado izquierdo:

$$m_1 a + T = m_1 g, \qquad m_2 a - T = -m_2 g$$

y de ahí se leen directamente los coeficientes.

In [ ]:
matriz_atwood = sp.Matrix([[m1, 1], [m2, -1]])
terminos_atwood = sp.Matrix([m1*g, -m2*g])

solucion_atwood = sp.simplify(matriz_atwood.solve(terminos_atwood))

aceleracion, tension = solucion_atwood
display(sp.Eq(sp.Symbol("a"), aceleracion))
display(sp.Eq(sp.Symbol("T"), tension))

Es el mismo resultado que la semana pasada con `linsolve`, escrito de otra
forma. La ventaja de la forma matricial no se nota con dos ecuaciones: se nota
con diez, y sobre todo cuando lo que queremos de la matriz **no** es resolver un
sistema sino diagonalizarla, que es lo que viene al final de la sesión.

Nota sobre las tres formas de resolver el mismo sistema:

| Escribe | Cuándo |
|---|---|
| `A.solve(b)` | Lo normal: $A$ cuadrada e invertible |
| `A.inv() * b` | Equivalente, pero calcula la inversa completa: más caro |
| `sp.linsolve((A, b), incognitas)` | Cuando el sistema puede ser degenerado o tener menos ecuaciones que incógnitas |

## TODO en clase 1

Una lámpara de peso $W$ cuelga de un nodo del que salen dos cables, uno hacia la
izquierda con ángulo $\alpha$ sobre la horizontal y otro hacia la derecha con
ángulo $\beta$. El equilibrio del nodo da dos ecuaciones, una por componente:

$$-T_1\cos\alpha + T_2\cos\beta = 0, \qquad T_1\sin\alpha + T_2\sin\beta = W$$

1. Declara `peso`, `alfa` y `beta` como símbolos positivos.
2. Arma `matriz_cables` (2×2, los coeficientes de $T_1$ y $T_2$) y
   `terminos_cables` (el vector columna del lado derecho).
3. Resuelve con `.solve()` y simplifica. Deben salirte
   $T_1 = W\cos\beta/\sin(\alpha+\beta)$ y su pareja.
4. Verifica sustituyendo de vuelta: `sp.simplify(matriz_cables*tensiones -
   terminos_cables)` tiene que ser `sp.zeros(2, 1)`.
5. Mira el determinante. ¿Para qué ángulos se anula, y qué configuración física
   es esa? (Pista: dibuja los dos cables en ese caso.)

In [ ]:
# TODO en clase: las tensiones de los dos cables
peso = ...
alfa = ...
beta = ...

matriz_cables = ...

terminos_cables = ...

tensiones = ...

## Eigenvalores y eigenvectores

Un **eigenvector** de $A$ es un vector al que $A$ no le cambia la dirección,
solo lo estira o lo encoge:

$$A\mathbf{v} = \lambda \mathbf{v}$$

y el factor $\lambda$ es su **eigenvalor**. SymPy los calcula simbólicamente:

| Quieres | Escribe | Devuelve |
|---|---|---|
| Solo los eigenvalores | `A.eigenvals()` | Diccionario `{valor: multiplicidad}` |
| Valores y vectores | `A.eigenvects()` | Lista de tuplas `(valor, multiplicidad, [vectores])` |
| El polinomio característico | `A.charpoly(lam).as_expr()` | Expresión en `lam` |
| La diagonalización | `A.diagonalize()` | El par `(P, D)` con $A = PDP^{-1}$ |

Ojo con lo que devuelve cada uno: un diccionario y una lista de tuplas son
objetos de Python, no expresiones — se recorren con un `for`, y lo que va por
`display` es la matriz o el eigenvalor de adentro.

In [ ]:
simetrica = sp.Matrix([[2, -1], [-1, 2]])

print("eigenvals:", simetrica.eigenvals())   # diccionario de Python

# eigenvects devuelve tuplas; se recorren, y cada pieza se muestra aparte.
for valor, multiplicidad, vectores in simetrica.eigenvects():
    print(f"eigenvalor (multiplicidad {multiplicidad}):")
    display(valor)
    for vector in vectores:
        display(vector.T)   # transpuesto solo para que ocupe un renglón

## Diagonalizar

Que una matriz sea diagonalizable significa que existe una base —la de sus
eigenvectores— en la que la matriz es diagonal. `A.diagonalize()` devuelve las
dos piezas: la matriz $P$ cuyas **columnas** son los eigenvectores, y la
diagonal $D$ con los eigenvalores en el orden correspondiente.

En física esa base tiene nombre propio según el problema: ejes principales de
inercia, modos normales de vibración, autoestados de un observable. Siempre es
lo mismo: la base en la que el problema se desacopla.

In [ ]:
P, D = simetrica.diagonalize()

display(P)
display(D)

# La definición, comprobada: A = P D P^-1.
display(
    sp.Eq(
        sp.Symbol("A - P D P^{-1}"),
        sp.simplify(P*D*P.inv() - simetrica),
        evaluate=False,
    )
)

## Física: dos masas y tres resortes

Dos masas iguales $m$ sobre una mesa sin fricción, unidas entre sí por un
resorte y a las paredes por otros dos, los tres de constante $k$. Si $x_1$ y
$x_2$ son los desplazamientos respecto al equilibrio, la segunda ley da

$$m\ddot{x}_1 = -k x_1 + k(x_2 - x_1), \qquad
  m\ddot{x}_2 = -k x_2 - k(x_2 - x_1)$$

que en forma matricial es $M\ddot{\mathbf{x}} = -K\mathbf{x}$, con

$$M = \begin{pmatrix} m & 0 \\ 0 & m\end{pmatrix}, \qquad
  K = \begin{pmatrix} 2k & -k \\ -k & 2k\end{pmatrix}$$

$K$ es la **matriz de rigidez**, y es simétrica: el resorte de en medio jala a
las dos masas por igual. Esa simetría no es casualidad, viene de que la fuerza
deriva de un potencial.

In [ ]:
rigidez = sp.Matrix([[2*k, -k], [-k, 2*k]])
masas = m * sp.eye(2)

display(sp.Eq(sp.Symbol("K"), rigidez, evaluate=False))
display(sp.Eq(sp.Symbol("M"), masas, evaluate=False))

print("¿K es simétrica?", rigidez.is_symmetric())

## La ecuación secular

Buscamos soluciones en las que las dos masas oscilan **con la misma frecuencia
y en fase fija**: $\mathbf{x}(t) = \mathbf{v}\cos(\omega t)$. Al sustituir,
las derivadas dan $-\omega^2$ y queda un problema puramente algebraico:

$$K\mathbf{v} = \omega^2 M \mathbf{v}$$

Es un problema de eigenvalores, con $\lambda = \omega^2$. Tiene solución no
trivial solo si la matriz $K - \lambda M$ es singular, es decir si

$$\det(K - \lambda M) = 0$$

Esa es la **ecuación secular**, y `solve` la resuelve como cualquier otra.

In [ ]:
lam = sp.Symbol("lambda", positive=True)

secular = sp.Eq(sp.det(rigidez - lam*masas), 0)
display(secular)

valores = sp.solve(secular, lam)
print("los dos valores de lambda = omega**2:")
for valor in valores:
    display(valor)

## Las frecuencias y los modos

Los eigenvalores son $\lambda = k/m$ y $\lambda = 3k/m$, así que las dos
frecuencias propias son $\omega = \sqrt{k/m}$ y $\omega = \sqrt{3k/m}$.

Falta lo interesante: **cómo** se mueve el sistema en cada una. Eso lo dicen los
eigenvectores de $M^{-1}K$, que es la misma ecuación $K\mathbf{v} = \lambda
M\mathbf{v}$ despejada.

In [ ]:
for valor, _, vectores in (masas.inv() * rigidez).eigenvects():
    print("omega =")
    display(sp.sqrt(valor))
    print("modo:")
    display(vectores[0].T)

Léelos como física:

- $\omega_1 = \sqrt{k/m}$ con modo $(1, 1)$: las dos masas se mueven **juntas,
  en fase**. El resorte de en medio nunca se estira, así que no participa — por
  eso la frecuencia es la de una sola masa con un solo resorte.
- $\omega_2 = \sqrt{3k/m}$ con modo $(-1, 1)$: las masas se mueven **en
  oposición**. El resorte central se comprime y estira al doble, y el sistema es
  más rígido: la frecuencia sube.

Que $\omega_2 > \omega_1$ era predecible sin cuenta alguna, y sirve de control.
Cualquier movimiento del sistema es una superposición de estos dos modos — lo
graficaremos en la sesión 2.

## TODO en clase 2

Repite el análisis con las **masas distintas**: $m_1$ y $m_2$, los tres resortes
todavía de constante $k$. La matriz de rigidez no cambia; la de masas sí.

1. Arma `masas_distintas` con `sp.diag(m1, m2)`.
2. Escribe la ecuación secular con `sp.det(rigidez - lam*masas_distintas)` y
   resuélvela. Van a salir dos raíces con un radical: no te asustes, es lo
   normal en cuanto se rompe la simetría.
3. Guarda las dos en `valores_distintos` y comprueba el caso conocido: al
   sustituir $m_2 = m_1$ y simplificar tienen que reaparecer $k/m_1$ y $3k/m_1$.
4. Toma el límite $m_2 \to \infty$ con `sp.limit` (semana 5) en la raíz menor.
   ¿Qué sistema físico es ese, y por qué el resultado tiene sentido?

El punto 3 es la costumbre que vale la pena llevarse de la semana: un resultado
simbólico nuevo se verifica reduciéndolo a uno que ya conocías.

In [ ]:
# TODO en clase: modos normales con masas distintas
masas_distintas = ...

secular_distintas = ...

valores_distintos = ...

## Resumen

Hoy pasamos de una ecuación a un sistema de ellas. `sp.Matrix` construye
matrices simbólicas; `*` es producto matricial y no conmuta; `det`, `inv` y
`rank` son simbólicos, y el determinante es el que avisa cuándo un sistema está
degenerado. Un sistema lineal se escribe $A\mathbf{x} = \mathbf{b}$ y se
resuelve con `.solve()`.

La segunda mitad fue el eigenproblema: `eigenvals`, `eigenvects` y
`diagonalize` devuelven fórmulas, no números, y en un sistema de osciladores
acoplados esas fórmulas son las **frecuencias propias** y los **modos
normales**. La ecuación secular $\det(K - \lambda M) = 0$ es un `solve` como
cualquier otro.

**Próxima sesión — Semana 6, sesión 2:** hasta ahora todo lo hemos leído como
fórmulas. Toca **verlo**: graficar con `sympy.plotting`, pasar a Matplotlib con
`lambdify` cuando hace falta control fino, y usar `sp.latex()` para escribir
reportes en los que los resultados se calculan, no se copian a mano.